# Программистская часть (4 балла)

## Natural Language Inference (2 балла)

Natural Language Inference — это задача "понимания" отношений между двумя утверждениями. Одно может следовать из другого (если (1) "все кошки полосатые" (2) "моя кошка полосатая", то из 1 следует 2 — но не наоборот), может противоречить ("все кошки полосатые" и "моя кошка не полосатая") или относиться нейтрально ("все кошки полосатые" и "моя кошка любит играть с клубком")

***Здесь и далее я буду писать свои внекодовые комментарии курсивом! Чтобы отличать их от основного задания***

Заведите модель, проверьте разные пары утверждений. Будет здорово, если получится найти что-то, на чем она отрабатывает неправильно — но это не обязательное требование для ДЗ.

In [1]:
# по техническим причинам, укажем AutoModelForSequenceClassification вместо просто AutoModel
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "cointegrated/rubert-base-cased-nli-threeway"  # русскоязычная модель для этой задачи
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text1 = "Сократ - человек, а все люди смертны."
text2 = "Сократ никогда не умрёт."

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:148: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--cointegrated--rubert-base-cased-nli-threeway. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Вызываем модель вот так. `return_tensors='pt'` говорит вернуть не список токенов, а так называемый тип `tensor`. Это что-то вроде `np.array`, но используется в нейросетях.

In [2]:
out = model(**tokenizer(text1, text2, return_tensors='pt'))

Посмотрите, что лежит в переменной out — результате вызова модели от токенов предложений

In [3]:
out

SequenceClassifierOutput(loss=None, logits=tensor([[-2.2051,  2.3795, -0.4113]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

`logits` — это то, что на самом деле отдают все нейросети. Их количество соответствует количеству классов. У нас три класса и один текст, поэтому мы получили матрицу размера одна строка на три числа. **Обратите внимание:** несмотря на то, что у нас одна пара текстов, мы всё равно получили *массив массивов*.

Логиты пропоцориональны вероятностям: чем больше значение, тем более вероятен класс.

Здесь полезно вспомнить логистическую регрессию: у нас есть число, а мы хотим его преобразовать его в диапазон от 0 до 1 — вероятность. Только теперь у нас целый массив чисел, и надо, чтобы они еще и суммировались к 1 (сумма вероятностей разных классов). Это делается называется **softmax** и делается так:

$s(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$

**Реализуйте функцию softmax**

In [49]:
import numpy as np
import math


def softmax(logits):  # logits is a list of lists, where each list has n elements! just a remimder for myself
    # You may want to convert logits to a NumPy array
    arr = logits.detach().numpy()

    # Compute the exponential of each logit   
    exps = np.exp(arr)
    # Compute the sum of exponentials for all logits
    sumexps = np.sum(exps)
    # Calculate softmax for each logit
    softmax_scores = exps/sumexps
    return softmax_scores

In [50]:
results = softmax(out.logits)
results

array([[0.00952592, 0.93320644, 0.05726764]], dtype=float32)

Сверьте вашу реализацию с готовой:

In [51]:
import torch

proba = torch.softmax(out.logits, -1)
proba

tensor([[0.0095, 0.9332, 0.0573]], grad_fn=<SoftmaxBackward0>)

***Кажется, у меня всё получилось :)***

In [52]:
# вот так можно получить список, а не tensor, если пользуетесь готовой реализацией
proba[0].tolist()

[0.009525923058390617, 0.9332063794136047, 0.05726763978600502]

У модели также параметр-словарь: из порядкового номера в "словесный" лейбл

In [53]:
model.config.id2label

{0: 'entailment', 1: 'contradiction', 2: 'neutral'}

Используйте словарь с названиями лейблов и посчитанные вероятности, чтобы получить словарь вида `{лейбл: вероятность}`

In [54]:
dict = {}
for x in results:
    dict["entailment"] = x[0]
    dict["contradiction"] = x[1]
    dict["neutral"] = x[2]
dict

{'entailment': 0.009525925, 'contradiction': 0.93320644, 'neutral': 0.05726764}

Самый вероятный класс для примера про Сократа был, наверное, ожидаем. Получилось ли у вас найти что-то, с чем модель не справилась?
**Ответ: *мне кажется, что иногда можно сломать модель на местоимении 1-го лица? Будто бы они заведомо думают, что говорящий -- человек... хотя это эксплицитно не говорится. См. пример внизу: мне кажется, будто это заведомо neutral утверждение, а модель считает, что это contradiction.*** 

In [56]:
text3 = "Я гусь, а гуси умеют летать"
text4 = "Люди не умеют летать"
attempt = model(**tokenizer(text3, text4, return_tensors='pt'))
resultsnew = softmax(attempt.logits)
dict1 = {}
for x in resultsnew:
    dict1["entailment"] = x[0]
    dict1["contradiction"] = x[1]
    dict1["neutral"] = x[2]
dict1

{'entailment': 0.024732055, 'contradiction': 0.7452178, 'neutral': 0.23005012}

## Анонимизация (2 балла)

Мы научились вызывать NER-тэггер. Воспользуйтесь семинаром, чтобы написать функцию, которая принимает некоторое предложение, а возвращает его же с удаленными named entity, замененными на их тэг. Например, `"Меня зовут Лена, я живу в Долгопрудном, работаю в Яндексе и преподаю в Вышке"` —>  `"Меня зовут <PER>, я живу в <LOC>, работаю в <LOC> и преподаю в <LOC>"`

1. ***Как артефакт прошлого оставляю здесь собственную функцию, которую я сделала до того, как я зашла в [документацию TokenClassificationPipeline](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.TokenClassificationPipeline) и увидела там информацию про аргумент entity_group... Она тоже работает, но, вероятно, медленнее, поскольку куча циклов.***

In [133]:
from transformers import pipeline
import re
classifier = pipeline("ner", model="Babelscape/wikineural-multilingual-ner")
line = "Меня зовут Лена, я живу в Долгопрудном, работаю в Яндексе и преподаю в Вышке"
def anonymizer(usersline):
    results = classifier(usersline)
    ents = []  # a list of entities
    coordinates = []  # a list of lists, each one of which I want to be a pair of starting and ending coord-s of ent-s in the string
    for i, el in enumerate(results):
        if re.match('B-...', el['entity']) is not None:  # if this is the beginning of the entity
            ents.append(el['entity'].split('-')[1])  # we secured the name of the entity
            coord1 = el['start']  # and its coordinates. if we're lucky, it will be the whole word
            coord2 = el['end']
            c = i + 1
            # now we have to check if the next few objects markes as entities are actually the parts of the same entity
            # lets'check if the starting point of the next entity (or the chain of them) is the same as ending of the preceding (so it is the
            # same entity and we habe to join it together). also have to be aware that list index isn't out of range
            while c < len(results) and results[c]['start'] == coord2:  
                coord2 = results[c]['end']
                c += 1  # checking until we have the whole entity captured
            coordinates.append([coord1, coord2])
    # now it is just about changing found substrings to corresponding entity tags
    words = []  # securing words that we have to change in the original line
    for x in coordinates:
        words.append(usersline[x[0]:x[1]])
    for i, el in enumerate(words):
        check = ents[i]
        usersline = usersline.replace(el, f'<{check}>')
    return usersline

In [134]:
anonymizer(line)

'Меня зовут <PER>, я живу в <LOC>, работаю в <LOC> и преподаю в <LOC>'

2. ***но при помощи аргумента grouped_entities всё делается вдвое быстрее!***

In [135]:
classifier = pipeline("ner", model="Babelscape/wikineural-multilingual-ner")
line = "Меня зовут Лена, я живу в Долгопрудном, работаю в Яндексе и преподаю в Вышке"
def norm_anonymizer(usersline):
    res = classifier(usersline, grouped_entities=True)
    for x in res:
        check = x['entity_group']
        usersline = usersline.replace(x['word'], f'<{check}>')
    return usersline

In [136]:
norm_anonymizer(line)

'Меня зовут <PER>, я живу в <LOC>, работаю в <LOC> и преподаю в <LOC>'

***обе функции работают!***

# Прикольная часть (3 балла)

Вы можете использовать BLOOM API - [bigscience/bloom](https://huggingface.co/bigscience/bloom) (окошко справа; нужно зарегистрироваться и залогиниться)

Для каждой задачи предоставьте краткое __описание__, как вы пробовали ее решить, и __скриншот__ из API.

__Например:__ Tony is talking to Darth Vader ([BLOOM API](https://huggingface.co/bigscience/bloom)). Черный текст — составленный промпт, синий — сгенерированный.

![img](https://i.imgur.com/a1QhKF7.png)

Ничего страшного, если вам придется несколько раз перезапустить генерацию. Но если вам приходится делать это больше 1-2 раз или у вас чаще получается ерунда, чем то, что хочется — имеет смысл попробовать переформулировать промпт.

## Диалог (1 балл)

**0.5 балла:** сгенерируйте диалог между любыми двумя персонажами из списка:
* любая знаменитость или историческая фигура
* любой персонаж книг/игр/другого медиа
* любое существо из мифологии или фольклора
* *вы*

***Я предоставляла модели описание того, что происходит (персонажей и ситуации -- conversation), а затем ставила имя персонажа с двоеточием, чтобы модель считывала начало диалога***

![img1](imgs/img1.png)

**0.5 балла:** то же самое (можно взять других персонажей, можно тех же), но теперь с дополнительной информацией о них или теме их разговора

***Здесь то же самое, но давала уточнение, о чём должна быть conversation***.

![img2](imgs/img2.png)
![img3](imgs/img3.png)

## Перевод (1 балл)

### ...слов

**0.25 балла:** составьте запрос к модели, переводящий любую строку любого стихтворения на другой язык.

***Здесь я давала instruction: "translate something" и так далее, при этом чётко обозначая элементом запроса "poem", что именно я хочу, чтобы перевелось***

![img4](imgs/img4.png)

**0.25 балла:** то же самое, но никак не формулируйте задачу, а только предоставьте несколько примеров (few-shot prompting)

***Просто накидала примеров стих + перевод, а затем дала модели только первую часть. Может быть, помогли многоточия, как маркеры конца фразы.***

![img5](imgs/img5.png)

### ...величин
**0.25 балла:** составьте запрос к модели, переводящий все метрические единицы в какой-либо фразе в имперские или наоборот. Помните, что 1 километр и 1 миля не равны.

***Я переводила из дюймов в сантимеры; отмечу, что слово "Answer" тоже в итоге написала я. Здесь ситуация сходная с пунктом про перевод, я давала инструкцию, давала данные, а затем начинала "Ответ...", чтобы модель не писала что-то другое, продолжая мой предыдущий промпт.***

![img6](imgs/img6.png)

**0.25 балла:** то же самое, но никак не формулируйте задачу, а только предоставьте несколько примеров (few-shot prompting)

***А здесь ответ получился уже правильнее! У меня лучше сработало, когда я поставила между предложениями знаки равно вместо просто переноса на новую строку, как я делала со стихами, может быть, это наталкивает на рассуждение, что речь идёт о счёте (даже при том, что я опечаталась в последнем примере...)***

![img7](imgs/img7.png)

## Ваш опыт (1 балл)

**0.5 балла**

Покажите пример самого интересного (на ваш взгляд) успешного взаимодействия с любой чат-моделью, которой вы пользовались. Возможно, модель помогала вам написать официально-деловое письмо на иностранном языке или придумать сюжет для фанфика?

Подойдет ChatGPT, Gemini, Bard или даже любые боты в телеграме. А если еще не пользовались — самое время попробовать!

***У меня нет опыта взаимодействия с такими моделями по рабочим вопросам, но несколько месяцев назад я в шутку спросила телеграм-бот ["ChatGPT-Unlimited"](https://t.me/gptunlimited_bot), чего было бы хорошо перекусить в процессе работы, чтобы это не было вредной едой, и модель дала несколько действительно неплохих вариантов, а после благодарности попрощалась в тему. Это, скорее всего, связано с тем, что этот бот отмечает отдельно, на какую тему с ним говорят, и выдаёт ответы в связи с этим.***
![img8](imgs/img8.png)

***Просто из забавного: когда-то в прошлом году я попробовала вместо того, чтобы генерировать картинку, спросить у чат-модели описание чего-нибудь словами. Получилось следующее:***
![img9](imgs/img9.png)
***смешно, конечно, но ведь модель правда неплохо справилась!***

**0.5 балла**

То же самое, но неудачное взаимодействие. С какой задачей модель так и не справилась, несмотря на ваши попытки?

***Конкретные примеры вспомнить сложно, но отмечу, что даже в процессе задания Bloom всё-таки не слишком справлялась с задачей переписывания предложения с одними единицами измерений на другие. Ещё могу добавить тот полупример, что та же модель ChatGPT-Unlimited допускает грамматические (лексические?) ошибки в генерации текста -- "преследит кругом орков" (я вдохновилась и пошла генерировать сюжеты для фанфиков...):***
![img10](imgs/img10.png)

# Ещё 5 баллов

Задача - предсказать POS-теги для каждого слова в тексте на основе эмбеддингов BERT.

Мы будем работать с данными в формате [Universal Dependencies](https://github.com/UniversalDependencies). Вы можете выбрать любой язык, для которого есть файлы train и dev/test.

In [8]:
train_filename = 'sa_vedic-ud-train.conllu'
test_filename = 'sa_vedic-ud-test.conllu'

Мы будем работать с форматом conll-U, в котором хранятся UD данные. Для обработки таких данных уже существуют готовые библиотеки. Нам нужно извлечь токены и pos-тэги. Напишем функцию:

In [9]:
import conllu
from tqdm.auto import tqdm

def get_dataset(filename):
    with open(filename) as f:
        data = f.read()
    text = []  # здесь мы будем хранить токены
    labels = []  # здесь мы будем хранить pos-тэги
    for sent in conllu.parse(data):
        tokens, pos = list(zip(*[(token["form"], token["upos"]) for token in sent if token["upos"] != "_"]))
        text.append(" ".join(tokens))  # нам важен контекст предложения
        labels.extend(pos)
    return text, labels

In [10]:
X_train, y_train = get_dataset(train_filename)
X_test, y_test = get_dataset(test_filename)

Теперь нам нужно получить эмбеддинги из модели. Мы будем работать с [мультилингвальным BERT](https://huggingface.co/google-bert/bert-base-multilingual-cased):

In [5]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
model = BertModel.from_pretrained("bert-base-multilingual-cased")

In [157]:
# model = model.to("cuda:0") # перенесем на GPU
# i'll use cpu instead: my torch installation argues that i din't have CUDA

AssertionError: Torch not compiled with CUDA enabled

Для начала напишем функцию, которая достает нам эмбеддинг предложения из модели:

In [11]:
def get_embedding(sentence, device="cpu"):
    inputs = tokenizer(sentence, padding=True, truncation=True,
                    return_tensors="pt").input_ids
    inputs = inputs.to(device)
    outputs = model(inputs)
    outputs = outputs["last_hidden_state"].detach().cpu().numpy().squeeze(0) # берем только последний слой
    return outputs

Как вы помните, BERT оперирует не на уровне слов, а на уровне word pieces, т. е. частей слов. Чтобы получить эмбеддинг токена, нам нужно объединить части слов. Нам нужны индексы токенов, которые принадлежат к одному и тому же токену. Мы должны получить список списков

In [49]:
def decode_tokens(sentence):
    tokens = []
    idx = 1
    for token in sentence.split():
        # токенизуруем один токен и получим части одного токена, подсказка: добавьте add_special_tokens=False
        encoded_word = tokenizer(token, add_special_tokens=False)
        tokparts = []  # let's store coordinates of tokens that are actually the same token here. it is close to what i did in punkt 1 of this hw...
        # напишите цикл, который возвращает список, где каждый токен представлен как список индексов его подчастей
        for x in encoded_word['input_ids']:  # checking every input_id
            tokparts.append(idx)  # we're beginning from the 1st position, because the 0 will always be 101
            idx += 1
            # so we're moving "the cursor" along the input_ids of the same word while it is this word
        tokens.append(tokparts)
    return tokens

In [103]:
sent = "Меня зовут Соня"

In [104]:
a = get_embedding(sent)
a

array([[ 0.12598406,  0.1331264 ,  0.39840996, ...,  0.3380909 ,
         0.04653442,  0.05562734],
       [-0.20336734,  0.35571545,  0.73442525, ...,  0.42489296,
        -0.3662591 , -0.4069993 ],
       [ 0.11835639,  0.7361884 ,  1.0480322 , ...,  0.7784686 ,
        -0.03746672,  0.21764895],
       ...,
       [ 0.47993958, -0.20241198,  0.3489251 , ...,  0.42529076,
        -0.2663645 ,  0.07395817],
       [ 0.3501139 , -0.14097829,  0.8666936 , ...,  0.56611097,
         0.04030976, -0.2941793 ],
       [ 0.3220774 ,  0.0538846 ,  0.9113989 , ...,  0.4885286 ,
        -0.01843214,  0.11274814]], dtype=float32)

In [62]:
# проверим, получился ли правильный список
assert len(decode_tokens(X_test[0])) == len(X_test[0].split())

Теперь напишем функцию, которая получает наш список списков с индексами и выдает **усредненный** эмбеддинг токена

In [135]:
import numpy as np

def get_embedding_per_token(tokens, sentence_embedding, length=768):
    embeddings = np.zeros((len(tokens), length))  # создадим пустую матрицу размеров токены х размер эмбеддингов
    for num, token in enumerate(tokens):
        try:
            embeddings[num,:] = np.mean(sentence_embedding[token[0]:(token[-1] + 1)], axis=0) # возьмите срез эмбеддинга предложения, равный нужному слову, и усредните
        except:
            embeddings[num,:] = np.zeros(768)
    return embeddings

Теперь давайте объединим все функции и напишем функцию, которая обрабатывает все предложения

In [136]:
def encode_dataset(X):
    embeddings = []
    for sentence in tqdm(X):
        sentence_embedding = get_embedding(sentence)
        tokens = decode_tokens(sentence)
        token_embeddings = get_embedding_per_token(tokens, sentence_embedding)
        embeddings.append(token_embeddings)
    return np.concatenate(embeddings)  # возвращаем матрицу

In [137]:
transformed_X_train = encode_dataset(X_train)
transformed_X_test = encode_dataset(X_test)

  0%|          | 0/2524 [00:00<?, ?it/s]

  0%|          | 0/1473 [00:00<?, ?it/s]

In [179]:
nan_mask = np.isnan(transformed_X_test)  # почему-то в тестовой выборке есть нан-элементы, из-за них вылетала ошибка. чистим их 
#  (ну, хотя бы заменяем на нули)
nan_mask

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [180]:
transformed_X_test_filtered = np.where(nan_mask, 0, transformed_X_test)

Теперь давайте обучим логистичесчкую регрессию на основе полученных эмбеддингов

In [181]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000)
logreg.fit(transformed_X_train, y_train)

LogisticRegression(max_iter=1000)

Теперь давайте измерим получившееся качество

In [182]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
y_pred = logreg.predict(transformed_X_test_filtered)
print("Accuracy is ", accuracy_score(y_test, y_pred))
print("F1-score is ", f1_score(y_test, y_pred, average="weighted"))

print(classification_report(y_test, y_pred))

Accuracy is  0.7127791563275434
F1-score is  0.7036988801673397
              precision    recall  f1-score   support

         ADJ       0.36      0.20      0.25       870
         ADV       0.80      0.77      0.78      1084
         AUX       0.57      0.51      0.54        90
       CCONJ       0.99      0.89      0.94       152
         DET       0.62      0.52      0.57        48
        NOUN       0.65      0.78      0.71      3074
         NUM       0.53      0.54      0.54        89
        PART       0.98      0.97      0.98       785
        PRON       0.80      0.78      0.79      1443
       SCONJ       0.83      0.72      0.77        97
        VERB       0.69      0.65      0.67      1940

    accuracy                           0.71      9672
   macro avg       0.71      0.67      0.69      9672
weighted avg       0.70      0.71      0.70      9672



Прокомментируйте получившиеся результаты

**Ваш ответ**: ого, всё даже не очень плохо! Можно обратить внимание, что лучше всего модель предсказывает союзы, частицы, местоимения и т.д., что логично, поскольку это закрытые лексические классы с фиксированными парадигмами, про которые модель быстро выучивает, что это они. Трудно дела обстоят со вспомогательными единицами, ещё хуже с прилагательными: это может быть связано с тем, что прилагательные в санскрите выглядят и ведут себя практически как существительные, а вспомогательные единицы могут быть грамматикализованы из каких-то лексем, в оригинале являющимися вполне себе самостоятельными и лексически наполненными, так что вспомогательные с ними мешаются (например, have в английском -- где-то часть словоформы, где-то самостоятельный лексический глагол). В среднем, всё логично, и метрики сравнительно неплохи!